<a href="https://colab.research.google.com/github/gleog6/course/blob/main/mt3/colab/music_transcription_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =================================================================
# 1. SETUP RESILIENTE (Nessun vincolo di versione bloccante)
# =================================================================
print("🧹 Pulizia e installazione forzata...")

# Definiamo subito le utility di Colab per evitare il NameError
from google.colab import files
import os, sys, gc, warnings

# Installazione dipendenze di sistema
!apt-get update -qq && apt-get install -qq libfluidsynth3 libasound2-dev > /dev/null

# Installazione librerie (senza bloccare la versione di TF che Google non trova più)
!pip install -qU numpy<1.24.0 numba==0.56.4 llvmlite==0.39.1
!pip install -qU essentia pydub mido soundfile pyfluidsynth omegaconf note-seq
!pip install -qU --no-deps magenta dopamine-rl dm-sonnet

# Patch per il bug delle nuove versioni di Python
!sed -i 's/import collections/import collections.abc as collections/g' /usr/local/lib/python3.10/dist-packages/magenta/models/shared/events_rnn_model.py 2>/dev/null || true

# Tentativo di importazione con gestione errori
try:
    import numpy as np
    import soundfile as sf
    import essentia.standard as es
    from pydub import AudioSegment
    from mido import MidiFile, MidiTrack
    import tensorflow.compat.v1 as tf
    tf.disable_v2_behavior() # Forza TF2 a comportarsi come TF1
    print("\n✅ AMBIENTE PRONTO: Essentia e TensorFlow configurati!")
except Exception as e:
    print(f"\n⚠️ Nota tecnica: {e}")
    print("L'ambiente è parziale, ma proviamo a procedere comunque.")

# =================================================================
# 2. LOGICA DI PULIZIA E TRASCRIZIONE
# =================================================================
def apply_essentia_cleaning(samples):
    # EQ per isolare le armoniche musicali
    eq = es.Equalizer(bands=[31, 62, 125, 250, 500, 1000, 2000, 4000, 8000, 16000],
                      gains=[-25, -10, 2, 3, 3, 3, 2, 0, -15, -30])
    return eq(samples)

print("\n--- CARICA IL TUO FILE AUDIO ---")
uploaded = files.upload() # Ora 'files' è definito all'inizio

if uploaded:
    fname = list(uploaded.keys())[0]
    audio = AudioSegment.from_file(fname).set_frame_rate(16000).set_channels(1)

    # Blocchi brevi per stabilità
    chunk_ms = 120000
    chunks = [audio[i:i + chunk_ms] for i in range(0, len(audio), chunk_ms)]

    if not os.path.exists("./checkpoints/mt3"):
        print("Download modello AI...")
        !gsutil -m cp -r gs://mt3/checkpoints .

    midis = []
    for i, chunk in enumerate(chunks):
        print(f"🔄 Trascrizione parte {i+1}/{len(chunks)}...")
        s = np.array(chunk.get_array_of_samples(), dtype=np.float32)
        s /= (np.max(np.abs(s)) + 1e-9)
        s = apply_essentia_cleaning(s)

        tmp_w, tmp_m = f"p{i}.wav", f"p{i}.mid"
        sf.write(tmp_w, s, 16000)

        # Inferenza AI via riga di comando (più stabile se gli import falliscono)
        !python3 -m magenta.models.mt3.mt3_inference \
            --checkpoint_path="./checkpoints/mt3" \
            --input_audio_path="{tmp_w}" \
            --output_midi_path="{tmp_m}"

        if os.path.exists(tmp_m):
            midis.append(tmp_m)
        if os.path.exists(tmp_w): os.remove(tmp_w)
        gc.collect()

    # =================================================================
    # 3. ASSEMBLAGGIO FINALE
    # =================================================================
    if midis:
        print("\nUnione file MIDI...")
        full_midi = MidiFile()
        for m_f in midis:
            m = MidiFile(m_f)
            for track in m.tracks:
                new_t = MidiTrack()
                for msg in track: new_t.append(msg)
                full_midi.tracks.append(new_t)

        full_midi.save("SPARTITO_ESSENTIA_UNITO.mid")
        files.download("SPARTITO_ESSENTIA_UNITO.mid")
        print("✅ Completato!")
    else:
        print("❌ L'AI non ha generato note. Verifica che il file audio non sia silenzioso.")